# Showcase: Protein Contact Network Graphicalizer

Small, explicit walk-through on five proteins. This notebook demonstrates the functions and graph artifacts without launching a bulk experiment.

## 1. Configure

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

ABSTRACTGRAPH_GRAPHICALIZER_SRC = PROJECT_ROOT.parent / "abstractgraph-ecosystem/repos/abstractgraph-graphicalizer/src"
if ABSTRACTGRAPH_GRAPHICALIZER_SRC.exists():
    sys.path.insert(0, str(ABSTRACTGRAPH_GRAPHICALIZER_SRC))

from enzyme_classification.data.protein_list import make_manifest_from_pdb_ids, write_manifest, read_manifest
from abstractgraph_graphicalizer.protein import ProteinContactNetworkGraphicalizer, download_mmcif
from abstractgraph_graphicalizer.protein import read_graph_pickle
from enzyme_classification.pipelines.build_graphs import build_graphs_from_manifest, summarize_status

MANIFEST_PATH = PROJECT_ROOT / "data/interim/showcase_6_proteins.tsv"
STATUS_PATH = PROJECT_ROOT / "data/processed/showcase/weighted_graph_build_status.tsv"
LOG_PATH = PROJECT_ROOT / "data/processed/showcase/weighted_graph_build_events.jsonl"
RAW_MMCIF_DIR = PROJECT_ROOT / "data/raw/mmcif"
GRAPH_DIR = PROJECT_ROOT / "data/processed/showcase/graphs_display_weighted"

MIN_DISTANCE = 4.0  # Ignore C-alpha pairs closer than this, mostly adjacent backbone neighbors.
MAX_DISTANCE = 8.0  # Keep C-alpha contacts up to this distance as spatial residue contacts.
SCALE_EDGE_WIDTH_BY_DISTANCE = True  # Draw shorter contacts with thicker edges and longer contacts with thinner edges.
MIN_EDGE_WIDTH = 0.25  # Width used for the longest contacts near MAX_DISTANCE.
MAX_EDGE_WIDTH = 3.8  # Width used for the shortest contacts near MIN_DISTANCE.
OVERWRITE = True  # Rebuild showcase graphs so changed display attributes are written into the cache.

## 2. Create And Display A 5-Protein Manifest

In [2]:
desired = make_manifest_from_pdb_ids(
    [
        "1ao6.A",  # Human serum albumin
        "4hhb.A",  # Hemoglobin alpha chain
        "1a3n.A",  # Hemoglobin-related human structure
        "1ubq.A",  # Ubiquitin protein chain
        "1hho.A",  # Human oxyhemoglobin alpha chain
        "1crn.A",  # Crambin compact protein chain
    ],
    protein_name="showcase candidate",
)
write_manifest(desired, MANIFEST_PATH)
desired

,sample_id,pdb_id,label_asym_id,auth_asym_id,protein_name
0,1ao6.A,1ao6,A,,showcase candidate
1,4hhb.A,4hhb,A,,showcase candidate
2,1a3n.A,1a3n,A,,showcase candidate
3,1ubq.A,1ubq,A,,showcase candidate
4,1hho.A,1hho,A,,showcase candidate
5,1crn.A,1crn,A,,showcase candidate


In [3]:
manifest = read_manifest(MANIFEST_PATH)
manifest

,sample_id,pdb_id,label_asym_id,auth_asym_id,protein_name
0,1ao6.A,1ao6,A,,showcase candidate
1,4hhb.A,4hhb,A,,showcase candidate
2,1a3n.A,1a3n,A,,showcase candidate
3,1ubq.A,1ubq,A,,showcase candidate
4,1hho.A,1hho,A,,showcase candidate
5,1crn.A,1crn,A,,showcase candidate


## 3. Explicit For Loop: Download And Persist Structures

In [4]:
for row in manifest.itertuples(index=False):
    try:
        path = download_mmcif(row.pdb_id, RAW_MMCIF_DIR, overwrite=False)
        print(f"{row.sample_id}: {path} ({path.stat().st_size:,} bytes)")
    except Exception as exc:
        print(f"{row.sample_id}: download error: {type(exc).__name__}: {exc}")

1ao6.A: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/raw/mmcif/1ao6.cif.gz (235,276 bytes)
4hhb.A: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/raw/mmcif/4hhb.cif.gz (187,621 bytes)
1a3n.A: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/raw/mmcif/1a3n.cif.gz (135,311 bytes)
1ubq.A: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/raw/mmcif/1ubq.cif.gz (26,915 bytes)
1hho.A: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/raw/mmcif/1hho.cif.gz (74,552 bytes)
1crn.A: /media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/raw/mmcif/1crn.cif.gz (17,580 bytes)


## 4. Invoke The Graphicalizer For One Sample

In [5]:
graphicalizer = ProteinContactNetworkGraphicalizer(
    mmcif_dir=RAW_MMCIF_DIR,
    graph_dir=GRAPH_DIR,
    min_distance=MIN_DISTANCE,
    max_distance=MAX_DISTANCE,
    scale_edge_width_by_distance=SCALE_EDGE_WIDTH_BY_DISTANCE,
    min_edge_width=MIN_EDGE_WIDTH,
    max_edge_width=MAX_EDGE_WIDTH,
    overwrite=OVERWRITE,
    return_graphs=False,
)

_, result = graphicalizer.transform_one(manifest.iloc[0].to_dict())
result

GraphBuildResult(sample_id='1ao6.A', pdb_id='1ao6', graph_path='/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/showcase/graphs_display_weighted/1ao6.A.pkl', status='created', message='')

## 5. Run The Resumable Pipeline On The Full Small List

In [6]:
status = build_graphs_from_manifest(
    MANIFEST_PATH,
    status_path=STATUS_PATH,
    log_path=LOG_PATH,
    raw_mmcif_dir=RAW_MMCIF_DIR,
    graph_dir=GRAPH_DIR,
    min_distance=MIN_DISTANCE,
    max_distance=MAX_DISTANCE,
    scale_edge_width_by_distance=SCALE_EDGE_WIDTH_BY_DISTANCE,
    min_edge_width=MIN_EDGE_WIDTH,
    max_edge_width=MAX_EDGE_WIDTH,
    overwrite=OVERWRITE,
    verbose=True,
)
summarize_status(status)

Starting graph build: rebuilding 6/6; 6/6 were already done
      row       done   elapsed       eta  status        sample        pdb     name
      1/6        0/6        0s   unknown  processing    1ao6.A        1ao6    showcase candidate
      1/6        1/6        0s        2s  created       1ao6.A        1ao6    showcase candidate
      2/6        1/6        0s        2s  processing    4hhb.A        4hhb    showcase candidate
      2/6        2/6        0s        1s  created       4hhb.A        4hhb    showcase candidate
      3/6        2/6        0s        1s  processing    1a3n.A        1a3n    showcase candidate
      3/6        3/6        1s        1s  created       1a3n.A        1a3n    showcase candidate
      4/6        3/6        1s        1s  processing    1ubq.A        1ubq    showcase candidate
      4/6        4/6        1s        0s  created       1ubq.A        1ubq    showcase candidate
      5/6        4/6        1s        0s  processing    1hho.A        1hho    sho

graph_status
done    6
Name: count, dtype: int64

## 6. Display Logs

In [7]:
import json

if LOG_PATH.exists():
    events = [json.loads(line) for line in LOG_PATH.read_text().splitlines() if line.strip()]
    for event in events[-20:]:
        print(
            f"[{event.get('ordinal', '-')}/{event.get('total', '-')}] "
            f"status={event.get('status')} done={event.get('done_count', '-')} "
            f"sample={event.get('sample_id')} name={event.get('protein_name')!r} "
            f"eta_seconds={event.get('eta_seconds')} message={event.get('message', '')}"
        )

[4/5] status=done done=5 sample=1ubq.A name='showcase candidate' eta_seconds=0.25315448849960376 message=created
[5/5] status=done done=5 sample=1hho.A name='showcase candidate' eta_seconds=0.0 message=created
[1/6] status=done done=1 sample=1ao6.A name='showcase candidate' eta_seconds=2.4533936850093596 message=created
[2/6] status=done done=2 sample=4hhb.A name='showcase candidate' eta_seconds=1.4008610760065494 message=created
[3/6] status=done done=3 sample=1a3n.A name='showcase candidate' eta_seconds=0.9074704250015203 message=created
[4/6] status=done done=4 sample=1ubq.A name='showcase candidate' eta_seconds=0.46925813500092767 message=created
[5/6] status=done done=5 sample=1hho.A name='showcase candidate' eta_seconds=0.2079175012004271 message=created
[6/6] status=done done=6 sample=1crn.A name='showcase candidate' eta_seconds=0.0 message=created
[1/6] status=done done=1 sample=1ao6.A name='showcase candidate' eta_seconds=3.9393160699546574 message=created
[2/6] status=done do

## 7. Inspect Persisted Graphs

In [8]:
done = status[status["graph_status"] == "done"].copy()
for row in done.itertuples(index=False):
    graph = read_graph_pickle(row.graph_path)
    print(f"{row.sample_id}: nodes={graph.number_of_nodes()} edges={graph.number_of_edges()} path={row.graph_path}")

1ao6.A: nodes=578 edges=2097 path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/showcase/graphs_display_weighted/1ao6.A.pkl
4hhb.A: nodes=141 edges=510 path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/showcase/graphs_display_weighted/4hhb.A.pkl
1a3n.A: nodes=141 edges=508 path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/showcase/graphs_display_weighted/1a3n.A.pkl
1ubq.A: nodes=76 edges=250 path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/showcase/graphs_display_weighted/1ubq.A.pkl
1hho.A: nodes=141 edges=511 path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/EnzymeClassification/data/processed/showcase/graphs_display_weighted/1hho.A.pkl
1crn.A: nodes=46 edges=158 path=/media/fabrizio/06bb7271-2161-43a4-91f1-98f

## 8. Graphical Display

Render all completed showcase PCNs with `abstractgraph.display.display_graphs`. For readability, each panel uses the largest connected component if the graph is disconnected.

In [ ]:
import math
import networkx as nx
import matplotlib.pyplot as plt

try:
    from abstractgraph.display import display_graph
except ImportError:
    ABSTRACTGRAPH_SRC = PROJECT_ROOT.parent / "abstractgraph-ecosystem/repos/abstractgraph/src"
    sys.path.insert(0, str(ABSTRACTGRAPH_SRC))
    from abstractgraph.display import display_graph

def edge_widths_for_display(graph, default=0.35):
    return [float(data.get("edge_width", default)) for _, _, data in graph.edges(data=True)]

def edge_width_summary(graph):
    widths = edge_widths_for_display(graph)
    missing = sum(1 for _, _, data in graph.edges(data=True) if "edge_width" not in data)
    if not widths:
        return "widths=none", missing
    return f"widths={min(widths):.2f}-{max(widths):.2f}", missing

def contact_bin_counts(graph):
    counts = {"contact_close": 0, "contact_far": 0, "contact": 0}
    for _, _, data in graph.edges(data=True):
        label = data.get("label", "contact")
        counts[label] = counts.get(label, 0) + 1
    return counts

done = status[status["graph_status"] == "done"].copy()
if len(done):
    display_items = []
    for row in done.itertuples(index=False):
        graph = read_graph_pickle(row.graph_path)
        components = sorted(nx.connected_components(graph), key=len, reverse=True)
        display_graph_obj = graph.subgraph(components[0]).copy() if components else graph
        counts = contact_bin_counts(display_graph_obj)
        width_text, missing_widths = edge_width_summary(display_graph_obj)
        if missing_widths:
            print(f"Warning: {row.sample_id} has {missing_widths} edges without edge_width; rebuild the showcase graph cache.")
        title = (
            f"{row.sample_id}  n={display_graph_obj.number_of_nodes()} e={display_graph_obj.number_of_edges()}\n"
            f"close={counts.get('contact_close', 0)} far={counts.get('contact_far', 0)}  {width_text}"
        )
        display_items.append((display_graph_obj, title))

    n_cols = 3
    n_rows = math.ceil(len(display_items) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(7.2 * n_cols, 5.4 * n_rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")

    print(f"Displaying {len(display_items)} completed graphs out of {len(status)} requested rows.")
    for ax, (graph, title) in zip(axes.ravel(), display_items):
        display_graph(
            graph,
            ax=ax,
            size=(7.2, 5.4),
            node_labels=False,
            edge_labels=False,
            style={
                "node_size": 20,
                "edge_width": edge_widths_for_display(graph),
                "node_alpha": 0.9,
                "edge_color": "#4f4f4f",
                "node_border_width": 0.1,
                "viewport_padding": 0.08,
            },
        )
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("No completed graphs available to display yet.")

Displaying 6 completed graphs out of 6 requested rows.
